# 🖼️ Daily Challenge: Classifying Handwritten Digits with CNNs

We build **two models** side by side and compare them:

| | Model A | Model B |
|---|---|---|
| **Type** | Fully Connected Neural Network (FCN) | Convolutional Neural Network (CNN) |
| **Input** | Flattened 784-dim vector | 28×28×1 image tensor |
| **Key layers** | Dense → Dense → Softmax | Conv2D → MaxPool → Conv2D → MaxPool → Dense → Softmax |
| **Expected accuracy** | ~98% | ~99%+ |

> **Runtime tip:** Runtime → Change runtime type → **T4 GPU** for faster training.

---
## 📦 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow : {tf.__version__}')
print(f'Keras      : {keras.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

---
## 1️⃣ Load the MNIST Dataset

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

print('── Raw shapes ──────────────────────────────────')
print(f'X_train : {X_train_raw.shape}   dtype: {X_train_raw.dtype}')
print(f'y_train : {y_train_raw.shape}   dtype: {y_train_raw.dtype}')
print(f'X_test  : {X_test_raw.shape}    dtype: {X_test_raw.dtype}')
print(f'y_test  : {y_test_raw.shape}    dtype: {y_test_raw.dtype}')
print()
print(f'Pixel value range : [{X_train_raw.min()}, {X_train_raw.max()}]')
print(f'Unique labels     : {np.unique(y_train_raw)}')

In [ ]:
# ── Visualize sample images ───────────────────────────────────────────────────
fig, axes = plt.subplots(3, 10, figsize=(16, 5))
fig.suptitle('MNIST — 3 samples per digit class', fontsize=13, fontweight='bold')

for digit in range(10):
    idx_list = np.where(y_train_raw == digit)[0][:3]
    for row, idx in enumerate(idx_list):
        ax = axes[row, digit]
        ax.imshow(X_train_raw[idx], cmap='gray_r')
        ax.set_title(f'{digit}', fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, labels, split in [(axes[0], y_train_raw, 'Train'), (axes[1], y_test_raw, 'Test')]:
    counts = np.bincount(labels)
    ax.bar(range(10), counts, color='steelblue', edgecolor='white')
    ax.set_xticks(range(10))
    ax.set_xlabel('Digit class')
    ax.set_ylabel('Count')
    ax.set_title(f'{split} set class distribution', fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    for x, c in enumerate(counts):
        ax.text(x, c + 30, str(c), ha='center', fontsize=7)
plt.tight_layout()
plt.show()
print('Classes are well-balanced — no resampling needed.')

---
## 2️⃣ Preprocess for the Fully Connected Network (FCN)

A Dense layer expects a **1-D vector** per sample, so we flatten 28×28 → 784.

In [ ]:
NUM_CLASSES = 10

# ── Flatten: (N, 28, 28) → (N, 784) ─────────────────────────────────────────
X_train_flat = X_train_raw.reshape(-1, 28 * 28).astype('float32')
X_test_flat  = X_test_raw.reshape(-1, 28 * 28).astype('float32')

# ── Normalize: [0, 255] → [0.0, 1.0] ────────────────────────────────────────
X_train_flat /= 255.0
X_test_flat  /= 255.0

# ── One-hot encode labels ─────────────────────────────────────────────────────
# keras.utils.np_utils.to_categorical is the same function as to_categorical
y_train_ohe = to_categorical(y_train_raw, NUM_CLASSES)
y_test_ohe  = to_categorical(y_test_raw,  NUM_CLASSES)

print('FCN preprocessing complete')
print(f'  X_train_flat : {X_train_flat.shape}   range [{X_train_flat.min():.1f}, {X_train_flat.max():.1f}]')
print(f'  X_test_flat  : {X_test_flat.shape}')
print(f'  y_train_ohe  : {y_train_ohe.shape}')
print(f'  y_test_ohe   : {y_test_ohe.shape}')
print(f'\nExample — label 5 as integer: {y_train_raw[0]}')
print(f'           label 5 one-hot  : {y_train_ohe[0]}')

In [ ]:
# ── Why flatten? Visualize the difference ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].imshow(X_train_raw[0], cmap='gray_r')
axes[0].set_title(f'Original 28×28 image  (label: {y_train_raw[0]})', fontweight='bold')
axes[0].axis('off')

axes[1].bar(range(784), X_train_flat[0], color='steelblue', width=1.0)
axes[1].set_title('Same image flattened to 784 pixel values', fontweight='bold')
axes[1].set_xlabel('Pixel index')
axes[1].set_ylabel('Normalized value')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('A Dense layer sees a flat vector — spatial structure (which pixel is next to which) is LOST.')

---
## 3️⃣ Build & Train the Fully Connected Neural Network (FCN)

In [ ]:
tf.random.set_seed(42)

# ── Architecture ──────────────────────────────────────────────────────────────
# Input (784) → Dense(256, ReLU) → Dense(128, ReLU) → Dense(10, Softmax)
fcn_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),             # regularization: randomly zero 20% of neurons
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')  # 10-class output
], name='FCN')

fcn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fcn_model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

history_fcn = fcn_model.fit(
    X_train_flat, y_train_ohe,
    epochs          = 20,
    batch_size      = 128,
    validation_split= 0.1,
    callbacks       = [early_stop],
    verbose         = 1
)

fcn_loss, fcn_acc = fcn_model.evaluate(X_test_flat, y_test_ohe, verbose=0)
print(f'\n── FCN Test Results ──────────────────')
print(f'   Loss     : {fcn_loss:.4f}')
print(f'   Accuracy : {fcn_acc*100:.2f}%')

---
## 4️⃣ Preprocess for the CNN

A `Conv2D` layer expects shape **(N, H, W, C)** — we add a channel dimension.

In [ ]:
# ── Reshape: (N, 28, 28) → (N, 28, 28, 1) ───────────────────────────────────
# The extra dimension = 1 grayscale channel (RGB images would have 3)
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32')  / 255.0

# Labels reuse the same one-hot arrays from FCN preprocessing

print('CNN preprocessing complete')
print(f'  X_train_cnn : {X_train_cnn.shape}   range [{X_train_cnn.min():.1f}, {X_train_cnn.max():.1f}]')
print(f'  X_test_cnn  : {X_test_cnn.shape}')
print()
print('Shape breakdown: (samples, height, width, channels)')
print('  60000 images, 28px tall, 28px wide, 1 grayscale channel')

In [ ]:
# ── How a Conv2D filter works — visualize one image's channels ───────────────
sample_img = X_train_cnn[0]   # shape (28, 28, 1)

# Simulate two hand-crafted edge-detection filters
h_edge = np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype='float32')  # horizontal edge
v_edge = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype='float32')  # vertical edge

def apply_filter(img2d, kernel):
    from scipy.signal import convolve2d
    return convolve2d(img2d, kernel, mode='same', boundary='fill')

img2d = sample_img[:, :, 0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(img2d, cmap='gray_r')
axes[0].set_title(f'Original (label: {y_train_raw[0]})', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(apply_filter(img2d, h_edge), cmap='gray')
axes[1].set_title('Horizontal edge filter output', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(apply_filter(img2d, v_edge), cmap='gray')
axes[2].set_title('Vertical edge filter output', fontweight='bold')
axes[2].axis('off')

plt.suptitle('What Conv2D filters detect (edges, textures, patterns)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('A CNN learns these filters automatically during training.')

---
## 5️⃣ Build & Train the CNN

In [ ]:
tf.random.set_seed(42)

# ── Architecture ──────────────────────────────────────────────────────────────
#
#  Input (28,28,1)
#    └─ Conv2D(32 filters, 3×3, ReLU)  → feature maps (28,28,32)
#         └─ MaxPool2D(2×2)             → (14,14,32)
#              └─ Conv2D(64, 3×3, ReLU) → (14,14,64)
#                   └─ MaxPool2D(2×2)   → (7,7,64)
#                        └─ Flatten     → 3136
#                             └─ Dense(128, ReLU) → Dropout → Dense(10, Softmax)

cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # ── Block 1: detect low-level features (edges, curves) ───────────────────
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
    layers.MaxPool2D(pool_size=(2, 2)),   # spatial downsampling: 28→14

    # ── Block 2: detect higher-level features (loops, corners) ───────────────
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
    layers.MaxPool2D(pool_size=(2, 2)),   # 14→7

    # ── Classifier head ──────────────────────────────────────────────────────
    layers.Flatten(),                     # (7,7,64) → 3136
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='CNN')

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
history_cnn = cnn_model.fit(
    X_train_cnn, y_train_ohe,
    epochs           = 15,
    batch_size       = 128,
    validation_split = 0.1,
    callbacks        = [early_stop],
    verbose          = 1
)

cnn_loss, cnn_acc = cnn_model.evaluate(X_test_cnn, y_test_ohe, verbose=0)
print(f'\n── CNN Test Results ──────────────────')
print(f'   Loss     : {cnn_loss:.4f}')
print(f'   Accuracy : {cnn_acc*100:.2f}%')

---
## 6️⃣ Compare FCN vs CNN Performance

In [ ]:
# ── Training curves: 4-panel comparison ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('FCN vs CNN — Training History', fontsize=14, fontweight='bold')

models_info = [
    ('FCN', history_fcn, 'steelblue', 'royalblue'),
    ('CNN', history_cnn, 'darkorange', 'tomato'),
]

for col, (name, hist, c1, c2) in enumerate(models_info):
    ep = range(1, len(hist.history['loss']) + 1)

    # Loss
    axes[0, col].plot(ep, hist.history['loss'],     color=c1, label='Train loss',      marker='o', markevery=2)
    axes[0, col].plot(ep, hist.history['val_loss'], color=c2, label='Val loss',        marker='s', markevery=2, linestyle='--')
    axes[0, col].set_title(f'{name} — Loss', fontweight='bold')
    axes[0, col].set_xlabel('Epoch')
    axes[0, col].set_ylabel('Loss')
    axes[0, col].legend()
    axes[0, col].grid(alpha=0.3)

    # Accuracy
    axes[1, col].plot(ep, hist.history['accuracy'],     color=c1, label='Train accuracy', marker='o', markevery=2)
    axes[1, col].plot(ep, hist.history['val_accuracy'], color=c2, label='Val accuracy',   marker='s', markevery=2, linestyle='--')
    axes[1, col].set_title(f'{name} — Accuracy', fontweight='bold')
    axes[1, col].set_xlabel('Epoch')
    axes[1, col].set_ylabel('Accuracy')
    axes[1, col].legend()
    axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Accuracy bar chart ────────────────────────────────────────────────────────
models_names  = ['FCN\n(Fully Connected)', 'CNN\n(Convolutional)']
test_accs     = [fcn_acc * 100, cnn_acc * 100]
param_counts  = [fcn_model.count_params(), cnn_model.count_params()]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('FCN vs CNN — Head-to-Head Comparison', fontsize=13, fontweight='bold')

# Accuracy
bars = axes[0].bar(models_names, test_accs, color=['steelblue', 'darkorange'], edgecolor='white', width=0.5)
axes[0].set_ylim(95, 100.5)
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Test Accuracy', fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')
for bar, acc in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{acc:.2f}%', ha='center', fontweight='bold', fontsize=12)

# Parameter count
bars2 = axes[1].bar(models_names, param_counts, color=['steelblue', 'darkorange'], edgecolor='white', width=0.5)
axes[1].set_ylabel('Total Parameters')
axes[1].set_title('Model Size (# parameters)', fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')
for bar, p in zip(bars2, param_counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f'{p:,}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

print(f'FCN — {param_counts[0]:,} params → {test_accs[0]:.2f}% accuracy')
print(f'CNN — {param_counts[1]:,} params → {test_accs[1]:.2f}% accuracy')
print(f'CNN gain: +{test_accs[1]-test_accs[0]:.2f}% accuracy with likely fewer parameters.')

In [ ]:
# ── Confusion matrices: FCN vs CNN ───────────────────────────────────────────
y_pred_fcn = np.argmax(fcn_model.predict(X_test_flat, verbose=0), axis=1)
y_pred_cnn = np.argmax(cnn_model.predict(X_test_cnn,  verbose=0), axis=1)
y_true     = y_test_raw

cm_fcn = confusion_matrix(y_true, y_pred_fcn)
cm_cnn = confusion_matrix(y_true, y_pred_cnn)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrices — FCN vs CNN', fontsize=13, fontweight='bold')

for ax, cm, name, cmap in [
    (axes[0], cm_fcn, 'FCN', 'Blues'),
    (axes[1], cm_cnn, 'CNN', 'Oranges')
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=range(10), yticklabels=range(10))
    ax.set_title(f'{name} Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')

plt.tight_layout()
plt.show()

In [ ]:
# ── Per-class accuracy: where does each model struggle? ──────────────────────
per_class_fcn = cm_fcn.diagonal() / cm_fcn.sum(axis=1) * 100
per_class_cnn = cm_cnn.diagonal() / cm_cnn.sum(axis=1) * 100

x_pos = np.arange(10)
width = 0.38

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x_pos - width/2, per_class_fcn, width, label='FCN', color='steelblue',   edgecolor='white')
ax.bar(x_pos + width/2, per_class_cnn, width, label='CNN', color='darkorange', edgecolor='white')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Digit {i}' for i in range(10)])
ax.set_ylim(92, 100.5)
ax.set_ylabel('Per-class accuracy (%)')
ax.set_title('Per-digit Accuracy: FCN vs CNN', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Classification report — FCN')
print(classification_report(y_true, y_pred_fcn, target_names=[str(i) for i in range(10)]))

print('Classification report — CNN')
print(classification_report(y_true, y_pred_cnn, target_names=[str(i) for i in range(10)]))

In [ ]:
# ── Visualize CNN feature maps on a sample image ─────────────────────────────
# See what the first Conv2D layer has learned to look for

sample = X_train_cnn[3:4]  # shape (1, 28, 28, 1)

feature_map_model = keras.Model(
    inputs  = cnn_model.input,
    outputs = cnn_model.layers[0].output   # first Conv2D output
)
feature_maps = feature_map_model.predict(sample, verbose=0)  # (1, 28, 28, 32)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(f'CNN Layer 1 Feature Maps for digit "{y_train_raw[3]}" — 32 learned filters',
             fontsize=12, fontweight='bold')

for i, ax in enumerate(axes.ravel()):
    ax.imshow(feature_maps[0, :, :, i], cmap='viridis')
    ax.set_title(f'f{i}', fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Each of the 32 feature maps highlights a different learned pattern (edges, curves, etc.).')

In [ ]:
# ── Misclassified samples — where CNN still fails ────────────────────────────
wrong_idx = np.where(y_pred_cnn != y_true)[0]
print(f'CNN misclassified {len(wrong_idx)} / {len(y_true)} test samples ({len(wrong_idx)/len(y_true)*100:.2f}%)')

sample_wrong = wrong_idx[:20]
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
fig.suptitle('CNN Misclassified Samples (true → predicted)', fontsize=12, fontweight='bold')

for ax, idx in zip(axes.ravel(), sample_wrong):
    ax.imshow(X_test_raw[idx], cmap='gray_r')
    ax.set_title(f'True:{y_true[idx]}  Pred:{y_pred_cnn[idx]}', color='red', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Even humans would struggle with some of these!')

---
## 📋 Summary: FCN vs CNN

### Why does the CNN outperform the FCN?

| Property | FCN (Dense only) | CNN (Conv + Pool + Dense) |
|---|---|---|
| **Spatial awareness** | ❌ None — flattening destroys layout | ✅ Preserves 2D structure of the image |
| **Parameter sharing** | ❌ Each pixel gets its own weights | ✅ One filter slides over the whole image |
| **Translation invariance** | ❌ A "5" in the top-left looks different from a "5" in the center | ✅ MaxPooling makes detection position-independent |
| **Feature hierarchy** | ❌ Flat mapping of pixels to classes | ✅ Edges → curves → digit parts → digit |
| **Typical MNIST accuracy** | ~98% | ~99%+ |

### Key concepts recap

**Conv2D** — A small filter (e.g. 3×3) slides across the image and produces a feature map. The network learns what patterns each filter should detect.

**MaxPool2D** — Takes the maximum value in each 2×2 patch, reducing spatial size by half. This makes the network robust to small shifts in position.

**Flatten** — Converts the 3-D feature volume (H×W×C) into a 1-D vector so Dense layers can process it.

**Dropout** — Randomly zeros out neurons during training, reducing overfitting.

**When to use which?**
- Use a **FCN** for tabular/structured data where spatial relationships don't exist.
- Use a **CNN** whenever the input has a spatial or sequential grid structure — images, audio spectrograms, video frames.